In [1]:
%pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 4.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 9.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 12.4 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
  Attempting uninstall: clickm╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/9 [httpcore2]
    Found existing installation: click 8.2.1━━━━━━━━━━━━━━━━━━ 3/9 [httpcore2]
    Uninstalling click-8.2.1:0m━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/9 [httpcore2]
      Successfully uninstalled click-8.2.1━━━━━━━━━━━━━━━━━━━━ 3/9 [httpcore2]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [pgmpy]32m8/9 [pgmpy]gface_hub]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import warnings
warnings.filterwarnings("ignore")   # hide library notices

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

# 1) The graph: each pair is an arrow (parent, child)
model = DiscreteBayesianNetwork([
    ("rain", "maintenance"),
    ("rain", "train"),
    ("maintenance", "train"),
    ("train", "appointment"),
])

# 2) P(Rain): no parents, so one column
rain = TabularCPD(
    variable="rain", variable_card=3,
    values=[[0.7],    # none
            [0.2],    # light
            [0.1]],   # heavy
    state_names={"rain": ["none", "light", "heavy"]})

# 3) P(Maintenance | Rain): one column for each value of rain
#                rain = none  light  heavy
maintenance = TabularCPD(
    variable="maintenance", variable_card=2,
    values=[[0.4, 0.2, 0.1],    # yes
            [0.6, 0.8, 0.9]],   # no
    evidence=["rain"], evidence_card=[3],
    state_names={"maintenance": ["yes", "no"],
                 "rain": ["none", "light", "heavy"]})

# 4) P(Train | Rain, Maintenance): one column for each (rain, maintenance) pair
#     (none,yes) (none,no) (light,yes) (light,no) (heavy,yes) (heavy,no)
train = TabularCPD(
    variable="train", variable_card=2,
    values=[[0.8, 0.9, 0.6, 0.7, 0.4, 0.5],    # on time
            [0.2, 0.1, 0.4, 0.3, 0.6, 0.5]],   # delayed
    evidence=["rain", "maintenance"], evidence_card=[3, 2],
    state_names={"train": ["on time", "delayed"],
                 "rain": ["none", "light", "heavy"],
                 "maintenance": ["yes", "no"]})

# 5) P(Appointment | Train): one column for each value of train
#                train = on time  delayed
appointment = TabularCPD(
    variable="appointment", variable_card=2,
    values=[[0.9, 0.6],    # attend
            [0.1, 0.4]],   # miss
    evidence=["train"], evidence_card=[2],
    state_names={"appointment": ["attend", "miss"],
                 "train": ["on time", "delayed"]})

# 6) Put the tables into the graph and check them
model.add_cpds(rain, maintenance, train, appointment)
print("Model is correct:", model.check_model())

Model is correct: True


In [2]:
# P(r, m, t, a) = P(r) x P(m | r) x P(t | r, m) x P(a | t)
p = (rain.get_value(rain="none")
     * maintenance.get_value(maintenance="no", rain="none")
     * train.get_value(train="on time", rain="none", maintenance="no")
     * appointment.get_value(appointment="attend", train="on time"))
print(round(p, 4))

0.3402


In [3]:
p = (rain.get_value(rain="heavy")
     * maintenance.get_value(maintenance="yes", rain="heavy")
     * train.get_value(train="delayed", rain="heavy", maintenance="yes")
     * appointment.get_value(appointment="miss", train="delayed"))
print(round(p, 4))

0.0024


In [4]:
from pgmpy.inference import VariableElimination
inference = VariableElimination(model)

def show(evidence):
    """Print P(X | evidence) for every node X that we did not observe."""
    for node in ["rain", "maintenance", "train", "appointment"]:
        if node in evidence:
            print(f"{node}: {evidence[node]}   (observed)")
            continue
        result = inference.query([node], evidence=evidence, show_progress=False)
        print(node)
        for value, prob in zip(result.state_names[node], result.values):
            print(f"    {value}: {prob:.4f}")

show({"train": "delayed"})

rain
    none: 0.4601
    light: 0.3005
    heavy: 0.2394
maintenance
    yes: 0.3662
    no: 0.6338
train: delayed   (observed)
appointment
    attend: 0.6000
    miss: 0.4000


In [5]:
show({"appointment": "miss"})

rain
    none: 0.6065
    light: 0.2392
    heavy: 0.1544
maintenance
    yes: 0.3441
    no: 0.6559
train
    on time: 0.4802
    delayed: 0.5198
appointment: miss   (observed)


In [6]:
show({"train": "delayed", "rain": "heavy"})

rain: heavy   (observed)
maintenance
    yes: 0.1176
    no: 0.8824
train: delayed   (observed)
appointment
    attend: 0.6000
    miss: 0.4000


In [ ]:
### Bayesian Network Lab – Written Answers

**Question 13: Why are the hand calculation and code results the same?**

Both methods use exact inference. The code performs the same calculations and normalization as the manual method, but more efficiently.

**Question 14: Why is P(appointment = miss | train = delayed) exactly 0.4?**

Appointment depends only on Train. Since we know the train is delayed, we can directly read the probability of 0.4 from the conditional probability table.

**Question 16: Why does the probability of heavy rain increase after missing the appointment?**

Missing the appointment makes a train delay more likely. A delayed train makes heavy rain more likely, so the probability of heavy rain increases.

**Question 18: Why does the probability of maintenance increase only slightly from 0.1 to 0.1176?**

Under heavy rain, the probability of a train delay is similar with maintenance (0.6) and without maintenance (0.5). Therefore, observing a delay provides only weak evidence for maintenance.

In [ ]:

### Step 3: Joint Probability (Likelihood)

Given: No rain, no maintenance, train on time, and appointment attended.

Formula:

P(r, m, t, a) = P(r) × P(m | r) × P(t | r, m) × P(a | t)

From the probability tables:

- P(rain = none) = 0.7
- P(maintenance = no | rain = none) = 0.6
- P(train = on time | rain = none, maintenance = no) = 0.9
- P(appointment = attend | train = on time) = 0.9

Calculation:

P = 0.7 × 0.6 × 0.9 × 0.9

**Final Answer: P = 0.3402**

---

### Step 4: Joint Probability (Bad Day)

Given: Heavy rain, maintenance yes, train delayed, and appointment missed.

Formula:

P(r, m, t, a) = P(r) × P(m | r) × P(t | r, m) × P(a | t)

From the probability tables:

- P(rain = heavy) = 0.1
- P(maintenance = yes | rain = heavy) = 0.1
- P(train = delayed | rain = heavy, maintenance = yes) = 0.6
- P(appointment = miss | train = delayed) = 0.4

Calculation:

P = 0.1 × 0.1 × 0.6 × 0.4

**Final Answer: P = 0.0024**

---

### Step 6: Inference by Enumeration

Given: The train is delayed.

Goal: Calculate P(rain = none | train = delayed).

We calculate the unnormalized probabilities by summing over maintenance = yes and maintenance = no.

**1. Rain = none**

P(none, delayed) = 0.7 × [(0.4 × 0.2) + (0.6 × 0.1)]

= 0.7 × (0.08 + 0.06)

= 0.7 × 0.14

= 0.098

**2. Rain = light**

P(light, delayed) = 0.2 × [(0.2 × 0.4) + (0.8 × 0.3)]

= 0.2 × (0.08 + 0.24)

= 0.2 × 0.32

= 0.064

**3. Rain = heavy**

P(heavy, delayed) = 0.1 × [(0.1 × 0.6) + (0.9 × 0.5)]

= 0.1 × (0.06 + 0.45)

= 0.1 × 0.51

= 0.051

**4. Normalization**

Total = 0.098 + 0.064 + 0.051

Total = 0.213

Divide each unnormalized probability by the total:

P(none | delayed) = 0.098 / 0.213 = 0.4601

P(light | delayed) = 0.064 / 0.213 = 0.3005

P(heavy | delayed) = 0.051 / 0.213 = 0.2394

Verification:

0.4601 + 0.3005 + 0.2394 = 1.0000

**Final Answer: P(rain = none | train = delayed) = 0.4601**

### Conclusion

The Bayesian Network calculations show that:

- Step 3: The joint probability is 0.3402 (34.02%).
- Step 4: The joint probability is 0.0024 (0.24%).
- Step 6: Given that the train is delayed, the probability of no rain decreases from 0.7 to approximately 0.4601.
